In [1]:
# Cell 1: Clean Installation without dependency conflicts
import subprocess
import sys

def check_package_version(package_name):
    try:
        result = subprocess.run([sys.executable, "-c", f"import {package_name}; print({package_name}.__version__)"],
                              capture_output=True, text=True)
        if result.returncode == 0:
            current_version = result.stdout.strip()
            print(f"✅ {package_name}: {current_version}")
            return True
        return False
    except:
        return False

print("🔍 Checking existing packages...")

# Check what we actually need to install
need_torch = not check_package_version("torch")
need_transformers = not check_package_version("transformers")
need_hf = not check_package_version("huggingface_hub")

# Install only what's missing - no version upgrades to avoid conflicts
if need_torch:
    print("📦 Installing PyTorch...")
    !pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
else:
    print("✅ PyTorch already available")

if need_transformers:
    print("📦 Installing Transformers...")
    !pip install -q transformers accelerate
else:
    print("✅ Transformers already available")

if need_hf:
    print("📦 Installing HuggingFace Hub...")
    !pip install -q huggingface_hub
else:
    print("✅ HuggingFace Hub already available")

# Only install missing packages without upgrading existing ones
!pip install -q openpyxl tqdm

print("✅ Cell 1: Clean installation complete!")

🔍 Checking existing packages...
✅ torch: 2.9.0+cu128
✅ transformers: 5.0.0
✅ huggingface_hub: 1.4.1
✅ PyTorch already available
✅ Transformers already available
✅ HuggingFace Hub already available
✅ Cell 1: Clean installation complete!


In [2]:
# Cell 2: Imports + Performance Settings + Model Selection (Fixed)
import os
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# Skip dynamo patch for compatibility
print(f"🔥 PyTorch version: {torch.__version__}")

from google.colab import drive
from huggingface_hub import notebook_login, login
from google.colab import userdata
import pandas as pd
import json
import time
import math
import re
import textwrap
import logging
import numpy as np
import torch._dynamo
torch._dynamo.config.cache_size_limit = 512
torch._dynamo.config.suppress_errors = True
try:
    from sklearn.metrics import cohen_kappa_score, confusion_matrix
    import matplotlib.pyplot as plt
    import seaborn as sns
    print("✅ sklearn/matplotlib available")
except ImportError:
    print("⚠️ sklearn/matplotlib not available - will skip advanced metrics")
from IPython.display import display
from tqdm import tqdm
from typing import Optional, Dict, Any, List, Tuple
import random

# CRITICAL: Performance optimizations BEFORE model loading
os.environ["GEMMA_DISABLE_SOFTCAP"] = "1"  # Critical for Gemma performance
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
logging.getLogger("transformers").setLevel(logging.ERROR)
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

# Set pandas display options
pd.set_option('display.max_colwidth', None)

# ADDED: Model configuration dictionary
MODEL_CONFIGS = {
    "gemma": {
        "model_id": "google/gemma-2-9b-it",
        "name": "Gemma2-9B-IT"
    },
    "llama": {
        "model_id": "meta-llama/Meta-Llama-3-8B-Instruct",
        "name": "Llama3-8B-Instruct"
    },
    "deepseek": {
        "model_id": "deepseek-ai/deepseek-llm-7b-chat",
        "name": "DeepSeek-7B-Chat"
    }
}

# ADDED: Set deterministic behavior for zero-shot
def set_deterministic_mode():
    """Set deterministic mode for reproducible zero-shot evaluation"""
    torch.manual_seed(42)
    torch.cuda.manual_seed(42)
    torch.cuda.manual_seed_all(42)
    np.random.seed(42)
    random.seed(42)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_deterministic_mode()

print("✅ Cell 2: Imports + performance settings + deterministic mode!")
print(f"🔥 GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"🎯 GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"🚀 Performance: GEMMA_DISABLE_SOFTCAP + TF32 + Deterministic mode enabled")
print(f"🤖 Available models: {list(MODEL_CONFIGS.keys())}")

🔥 PyTorch version: 2.9.0+cu128
✅ sklearn/matplotlib available
✅ Cell 2: Imports + performance settings + deterministic mode!
🔥 GPU Available: True
🎯 GPU Name: NVIDIA RTX PRO 6000 Blackwell Server Edition
🚀 Performance: GEMMA_DISABLE_SOFTCAP + TF32 + Deterministic mode enabled
🤖 Available models: ['gemma', 'llama', 'deepseek']


/usr/local/lib/python3.12/dist-packages/torch/backends/__init__.py:46: UserWarning: Please use the new API settings to control TF32 behavior, such as torch.backends.cudnn.conv.fp32_precision = 'tf32' or torch.backends.cuda.matmul.fp32_precision = 'ieee'. Old settings, e.g, torch.backends.cuda.matmul.allow_tf32 = True, torch.backends.cudnn.allow_tf32 = True, allowTF32CuDNN() and allowTF32CuBLAS() will be deprecated after Pytorch 2.9. Please see https://pytorch.org/docs/main/notes/cuda.html#tensorfloat-32-tf32-on-ampere-and-later-devices (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:80.)
  self.setter(val)


In [3]:
# Cell 3: Mount Drive & Load Data
drive.mount('/content/drive')

file_path = '/content/drive/MyDrive/Zeroshot/Data.xlsx'
print("📂 Loading dataset...")

try:
    df = pd.read_excel(file_path)
    print(f"✅ Dataset loaded: {len(df):,} records")
    print(f"📋 Columns: {list(df.columns)}")

    # Data is already in correct scale (0-2, 0-4) from preprocessing
    df_adjusted = df.copy()

    print(f"\n📊 Score Ranges (should match fine-tuning):")
    print(f"   Clarity: {df['Clarity_Score'].min()}-{df['Clarity_Score'].max()} (expected 0-2)")
    print(f"   Terminology: {df['Terminology_Score'].min()}-{df['Terminology_Score'].max()} (expected 0-2)")
    print(f"   Coverage: {df['Coverage_Score'].min()}-{df['Coverage_Score'].max()} (expected 0-2)")
    print(f"   Accuracy: {df['Accuracy_Score'].min()}-{df['Accuracy_Score'].max()} (expected 0-4)")

    print(f"\n📊 Data overview:")
    print(f"   Total records: {len(df):,}")

    # Check grade distribution
    print(f"\n📈 Final grade distribution:")
    print(df['Rubric_Grade'].value_counts().sort_index())

except Exception as e:
    print(f"❌ Error loading data: {e}")
    print("💡 Make sure Data.xlsx is in /content/drive/MyDrive/Zeroshot/")

print("\n✅ Data loading complete!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
📂 Loading dataset...
✅ Dataset loaded: 2,550 records
📋 Columns: ['ID', 'Question', 'Model Answer', 'Answer Text', 'Clarity_Score', 'Terminology_Score', 'Coverage_Score', 'Accuracy_Score', 'Rubric_Grade', 'off_topic', 'Off_Topic_Reason']

📊 Score Ranges (should match fine-tuning):
   Clarity: 0-2 (expected 0-2)
   Terminology: 0-2 (expected 0-2)
   Coverage: 0-2 (expected 0-2)
   Accuracy: 0-4 (expected 0-4)

📊 Data overview:
   Total records: 2,550

📈 Final grade distribution:
Rubric_Grade
0.0    468
0.5     30
1.0    184
1.5     64
2.0    198
2.5    206
3.0    199
3.5    199
4.0    286
4.5    258
5.0    458
Name: count, dtype: int64

✅ Data loading complete!


In [4]:
# Cell 4: Interactive Model Selection & Loading
print("🤖 CHOOSE YOUR MODEL:")
print("1. gemma    - Gemma2-9B-IT")
print("2. llama    - Llama3-8B-Instruct")
print("3. deepseek - DeepSeek-7B-Chat")

# Interactive model selection
model_choice = input("\nEnter your choice (gemma/llama/deepseek): ").strip().lower()

if model_choice not in MODEL_CONFIGS:
    print("❌ Invalid choice. Defaulting to gemma")
    model_choice = "gemma"

SELECTED_MODEL = model_choice
print(f"\n🔄 Loading {MODEL_CONFIGS[SELECTED_MODEL]['name']}...")

# HuggingFace Authentication
print("🔑 Authenticating with Hugging Face...")
try:
    HUGGING_FACE_TOKEN = userdata.get('HF_TOKEN')
    if HUGGING_FACE_TOKEN is None:
        print("Using manual login...")
        notebook_login()
    else:
        login(token=HUGGING_FACE_TOKEN)
        print("✅ Logged in with saved token!")
except Exception as e:
    print(f"Using manual login: {e}")
    notebook_login()

# Load selected model
model_id = MODEL_CONFIGS[SELECTED_MODEL]["model_id"]
dtype = torch.bfloat16

print(f"🤖 Loading {model_id}...")

# Clear cache before loading
torch.cuda.empty_cache()

try:
    # Load tokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_id)

    # Set pad token if needed
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    # Load model with optimal settings
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype=dtype,
        device_map="auto",
        attn_implementation="sdpa"
    ).eval()

    # Warm-up call
    print("🔥 Warming up model...")
    _ = model.generate(**tokenizer("warm-up", return_tensors="pt").to(model.device), max_new_tokens=1)

    print(f"✅ {MODEL_CONFIGS[SELECTED_MODEL]['name']} loaded successfully!")
    print(f"🎯 Device: {next(model.parameters()).device}")
    print(f"🧠 Attention: {model.config._attn_implementation}")
    print(f"💾 Memory: {torch.cuda.memory_allocated(0) / 1024**3:.1f} GB")

except Exception as e:
    print(f"❌ Error loading model: {e}")

print("✅ Cell 4: Model loading complete!")

🤖 CHOOSE YOUR MODEL:
1. gemma    - Gemma2-9B-IT
2. llama    - Llama3-8B-Instruct
3. deepseek - DeepSeek-7B-Chat

Enter your choice (gemma/llama/deepseek): deepseek

🔄 Loading DeepSeek-7B-Chat...
🔑 Authenticating with Hugging Face...
✅ Logged in with saved token!
🤖 Loading deepseek-ai/deepseek-llm-7b-chat...


config.json:   0%|          | 0.00/594 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

pytorch_model.bin.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Loading weights:   0%|          | 0/273 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

🔥 Warming up model...
✅ DeepSeek-7B-Chat loaded successfully!
🎯 Device: cuda:0
🧠 Attention: sdpa
💾 Memory: 12.9 GB
✅ Cell 4: Model loading complete!


In [5]:
# Cell 5: Updated Prompt Template (Matched with Fine-tuning + Off-Topic Detection)
UPDATED_PROMPT_TEMPLATE = """You are an expert AI Teaching Assistant evaluating student responses for accounting questions. Grade objectively using the provided rubric.

**QUESTION:** {question_text}

**MODEL ANSWER (Benchmark):** {model_answer_text}

**STUDENT ANSWER:** {student_answer_text}

**GRADING RUBRIC:**
- **Clarity** (0-2): Answer clarity and directness IF relevant to question
- **Terminology** (0-2): Correct business/accounting terms IF relevant
- **Coverage** (0-2): Addresses main points from model answer IF relevant
- **Accuracy** (0-4): Factual correctness of covered points IF relevant

**CRITICAL:** If the student answer is off-topic, doesn't address the question, or is left blank, set Off_Topic to true and assign 0 for ALL criteria.

**OUTPUT FORMAT:** Respond with ONLY a valid JSON object (no other text):
{{
  "Off_Topic": true/false,
  "Clarity_Score": X,
  "Terminology_Score": X,
  "Coverage_Score": X,
  "Accuracy_Score": X
}}"""

# Helper functions
def clean_model_answer_text(text_input):
    """Clean and format model answer text"""
    if not isinstance(text_input, str):
        return "" if pd.isna(text_input) else str(text_input)

    text = text_input
    text = re.sub(r'<br\s*/?>', '\n', text, flags=re.IGNORECASE)
    text = re.sub(r'[•¥]\s*\d*\s*', '- ', text)
    text = re.sub(r'\s+\d+\s*$', '', text)
    lines = [line.strip() for line in text.splitlines()]
    cleaned_text = "\n".join(line for line in lines if line)
    return cleaned_text.strip()

def parse_json_response(response: str) -> Optional[Dict]:
    """Parse JSON response - handles malformed JSON with regex fallback"""
    if not response:
        return None

    # Remove markdown code blocks
    response = response.replace('```json', '').replace('```', '').strip()

    # Try direct parsing first
    try:
        return json.loads(response)
    except json.JSONDecodeError:
        pass

    # Try extracting JSON from text
    try:
        start = response.find('{')
        end = response.rfind('}') + 1
        if start != -1 and end > start:
            json_part = response[start:end]
            return json.loads(json_part)
    except json.JSONDecodeError:
        pass

    # Regex fallback for malformed JSON
    scores = {"Off_Topic": False, "Clarity_Score": 0, "Terminology_Score": 0, "Coverage_Score": 0, "Accuracy_Score": 0}
    for key in ["Clarity_Score", "Terminology_Score", "Coverage_Score", "Accuracy_Score"]:
        pattern = rf'"{key}"?\s*:\s*(\d+)'
        match = re.search(pattern, response)
        if match:
            scores[key] = int(match.group(1))

    # Extract Off_Topic boolean
    ot_match = re.search(r'"Off_Topic"?\s*:\s*(true|false)', response, re.IGNORECASE)
    if ot_match:
        scores["Off_Topic"] = ot_match.group(1).lower() == "true"

    # Check if any scores were found
    if any(v for k, v in scores.items() if k != "Off_Topic"):
        return scores

    return None

def calculate_final_grade(scores: Dict) -> Optional[float]:
    """Calculate final grade = (C + T + V + A) / 2"""
    try:
        c = scores.get('Clarity_Score', 0)
        t = scores.get('Terminology_Score', 0)
        v = scores.get('Coverage_Score', 0)
        a = scores.get('Accuracy_Score', 0)
        return (c + t + v + a) / 2
    except:
        return None

print("✅ Prompt template updated with Off-Topic detection!")
print("   Scales: Clarity (0-2), Terminology (0-2), Coverage (0-2), Accuracy (0-4)")
print("   Output: JSON with Off_Topic + scores")

✅ Prompt template updated with Off-Topic detection!
   Scales: Clarity (0-2), Terminology (0-2), Coverage (0-2), Accuracy (0-4)
   Output: JSON with Off_Topic + scores


In [6]:
# Cell 6: Clean Assessment Function (Updated)
def zero_shot_assessment(question_text: str, student_answer: str, model_answer: str,
                        model, tokenizer) -> Tuple[Optional[str], Optional[Dict]]:
    """Zero-shot assessment function - matched with fine-tuning settings"""
    try:
        prompt_content = UPDATED_PROMPT_TEMPLATE.format(
            question_text=str(question_text).strip(),
            model_answer_text=clean_model_answer_text(str(model_answer)),
            student_answer_text=str(student_answer).strip()
        )

        model_name = tokenizer.name_or_path.lower()

        if "gemma" in model_name:
            prompt_str = f"<start_of_turn>user\n{prompt_content}<end_of_turn>\n<start_of_turn>model\n"
        elif "deepseek" in model_name:
            prompt_str = f"User: {prompt_content}\n\nAssistant: "
        elif "llama" in model_name:
            try:
                messages = [{"role": "user", "content": prompt_content}]
                prompt_str = tokenizer.apply_chat_template(
                    messages, tokenize=False, add_generation_prompt=True
                )
            except:
                prompt_str = f"<|begin_of_text|><|start_header_id|>user<|end_header_id|>\n\n{prompt_content}<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n"
        else:
            prompt_str = f"User: {prompt_content}\n\nAssistant: "

        inputs = tokenizer(
            prompt_str,
            return_tensors="pt",
            truncation=True,
            max_length=2048
        ).to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                input_ids=inputs.input_ids,
                attention_mask=inputs.attention_mask,
                max_new_tokens=400,
                do_sample=False,
                temperature=0.0,
                pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
                eos_token_id=tokenizer.eos_token_id,
                use_cache=True
            )

        input_length = inputs.input_ids.shape[1]
        generated_tokens = outputs[0][input_length:]
        response = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

        # Parse response
        parsed = parse_json_response(response)

        return response, parsed

    except Exception as e:
        print(f"Assessment error: {e}")
        return None, None

print("✅")

✅


In [7]:
# Cell 7: Results Formatting (Updated)
def format_assessment_result(raw_response: str, parsed_json: Dict, human_scores: Dict) -> Dict:
    """Format assessment result for comparison"""

    if parsed_json is None:
        return {
            'success': False,
            'raw_response': raw_response,
            'llm_scores': None,
            'human_scores': human_scores,
            'final_grade_llm': None,
            'final_grade_human': None,
            'error': 'Failed to parse response'
        }

    # Extract LLM scores
    llm_c = parsed_json.get('Clarity_Score', 0)
    llm_t = parsed_json.get('Terminology_Score', 0)
    llm_v = parsed_json.get('Coverage_Score', 0)
    llm_a = parsed_json.get('Accuracy_Score', 0)

    # Human scores
    h_c = human_scores.get('Clarity_Score', 0)
    h_t = human_scores.get('Terminology_Score', 0)
    h_v = human_scores.get('Coverage_Score', 0)
    h_a = human_scores.get('Accuracy_Score', 0)

    # Calculate final grades
    final_llm = (llm_c + llm_t + llm_v + llm_a) / 2
    final_human = (h_c + h_t + h_v + h_a) / 2

    return {
        'success': True,
        'raw_response': raw_response,
        'llm_scores': {
            'Clarity_Score': llm_c,
            'Terminology_Score': llm_t,
            'Coverage_Score': llm_v,
            'Accuracy_Score': llm_a
        },
        'human_scores': human_scores,
        'final_grade_llm': final_llm,
        'final_grade_human': final_human,
        'diff': abs(final_llm - final_human)
    }

print("✅ Results formatting updated!")

✅ Results formatting updated!


In [8]:
# Cell 8: Batch Processing Setup
print("🚀 BATCH PROCESSING SETUP")
print("="*50)

print("Choose your processing scope:")
print("1. Single question (30 students) - Fast testing")
print("2. Sample size (custom number) - Medium testing")
print("3. Full dataset (2,550 records) - Complete evaluation")

scope_choice = input("\nEnter choice (1/2/3): ").strip()

if scope_choice == "1":
    # Single question processing
    print("\n📋 Available questions (Q1-Q85)")
    question_num = input("Enter question number (e.g., '1' for Q1): ").strip()

    if question_num.isdigit():
        q_filter = f"Q{question_num}"
        filtered_df = df_adjusted[df_adjusted['ID'].str.startswith(q_filter)]

        if len(filtered_df) > 0:
            print(f"✅ Found {len(filtered_df)} records for {q_filter}")
            print(f"📊 Processing {q_filter} with {len(filtered_df)} student answers...")

            # Set processing parameters
            PROCESSING_DF = filtered_df
            BATCH_SIZE = len(filtered_df)  # Process all at once for single question
            OUTPUT_FILE = f"/content/drive/MyDrive/results_{MODEL_CONFIGS[SELECTED_MODEL]['name']}_Q{question_num}.csv"

        else:
            print(f"❌ No records found for Q{question_num}")
            PROCESSING_DF = None
    else:
        print("❌ Invalid question number")
        PROCESSING_DF = None

elif scope_choice == "2":
    # Sample processing
    sample_size = input("Enter sample size (e.g., 100): ").strip()

    if sample_size.isdigit():
        sample_size = int(sample_size)
        if sample_size <= len(df_adjusted):
            PROCESSING_DF = df_adjusted.sample(n=sample_size, random_state=42)
            BATCH_SIZE = min(50, sample_size)  # Process in chunks of 50
            OUTPUT_FILE = f"/content/drive/MyDrive/results_{MODEL_CONFIGS[SELECTED_MODEL]['name']}_sample_{sample_size}.csv"

            print(f"✅ Random sample of {sample_size} records selected")
            print(f"📊 Will process in batches of {BATCH_SIZE}")
        else:
            print(f"❌ Sample size too large. Maximum: {len(df_adjusted)}")
            PROCESSING_DF = None
    else:
        print("❌ Invalid sample size")
        PROCESSING_DF = None

elif scope_choice == "3":
    # Full dataset processing
    confirm = input("⚠️  Full dataset (2,550 records) will take ~5-6 hours. Continue? (y/n): ").strip().lower()

    if confirm == 'y':
        PROCESSING_DF = df_adjusted
        BATCH_SIZE = 100  # Process in chunks of 100
        OUTPUT_FILE = f"/content/drive/MyDrive/results_{MODEL_CONFIGS[SELECTED_MODEL]['name']}_full_dataset.csv"

        print(f"✅ Full dataset processing confirmed")
        print(f"📊 Will process {len(PROCESSING_DF)} records in batches of {BATCH_SIZE}")
        print(f"⏱️  Estimated time: {len(PROCESSING_DF) * 7 / 3600:.1f} hours")
    else:
        print("❌ Full processing cancelled")
        PROCESSING_DF = None

else:
    print("❌ Invalid choice")
    PROCESSING_DF = None

if PROCESSING_DF is not None:
    print(f"\n🎯 PROCESSING SUMMARY:")
    print(f"   Model: {MODEL_CONFIGS[SELECTED_MODEL]['name']}")
    print(f"   Records: {len(PROCESSING_DF):,}")
    print(f"   Batch size: {BATCH_SIZE}")
    print(f"   Output file: {OUTPUT_FILE}")
    print(f"   Estimated time: {len(PROCESSING_DF) * 7 / 60:.0f} minutes")

    # Show sample of what will be processed
    print(f"\n📋 Sample records to process:")
    print(PROCESSING_DF[['ID', 'Rubric_Grade']].head())

    print(f"\n✅ Ready for batch processing! Run Cell 9 to start.")
else:
    print(f"\n❌ No processing configured. Please run Cell 8 again.")

print("✅ Cell 8: Batch processing setup complete!")

🚀 BATCH PROCESSING SETUP
Choose your processing scope:
1. Single question (30 students) - Fast testing
2. Sample size (custom number) - Medium testing
3. Full dataset (2,550 records) - Complete evaluation

Enter choice (1/2/3): 3
⚠️  Full dataset (2,550 records) will take ~5-6 hours. Continue? (y/n): y
✅ Full dataset processing confirmed
📊 Will process 2550 records in batches of 100
⏱️  Estimated time: 5.0 hours

🎯 PROCESSING SUMMARY:
   Model: DeepSeek-7B-Chat
   Records: 2,550
   Batch size: 100
   Output file: /content/drive/MyDrive/results_DeepSeek-7B-Chat_full_dataset.csv
   Estimated time: 298 minutes

📋 Sample records to process:
            ID  Rubric_Grade
0  Q1 Student1           2.0
1  Q1 Student2           2.5
2  Q1 Student3           4.0
3  Q1 Student4           5.0
4  Q1 Student5           1.5

✅ Ready for batch processing! Run Cell 9 to start.
✅ Cell 8: Batch processing setup complete!


In [9]:
# Cell 9: Batch Processing Functions (Updated with Off-Topic Detection)
def batch_processing(df: pd.DataFrame, model, tokenizer,
                    start_idx: int = 0, end_idx: Optional[int] = None,
                    save_every: int = 50, output_path: str = None) -> pd.DataFrame:
    """Main batch processing function with progress tracking and auto-save"""

    if end_idx is None:
        end_idx = len(df)

    total_records = end_idx - start_idx
    print(f"🚀 Batch Processing: {total_records} records")
    print(f"💾 Auto-saving every {save_every} records")

    torch.cuda.empty_cache()
    print("🧹 GPU cache cleared before processing")

    results = []
    correct_1 = 0  # Within ±1.0
    correct_05 = 0  # Within ±0.5
    start_time = time.time()

    pbar = tqdm(range(start_idx, end_idx), desc="🚀 Processing")

    for idx in pbar:
        try:
            row = df.iloc[idx]

            student_id = str(row.get('ID', f'Record_{idx}'))
            question = str(row.get('Question', ''))
            student_answer = str(row.get('Answer Text', ''))
            model_answer = str(row.get('Model Answer', ''))

            # Human scores
            h_c = int(row.get('Clarity_Score', 0))
            h_t = int(row.get('Terminology_Score', 0))
            h_v = int(row.get('Coverage_Score', 0))
            h_a = int(row.get('Accuracy_Score', 0))
            human_grade = (h_c + h_t + h_v + h_a) / 2
            h_ot = str(row.get('off_topic', 'On-Topic')) != 'On-Topic'

            elapsed = time.time() - start_time
            rate = len(results) / max(elapsed, 1e-6)

            # Generate assessment
            record_start = time.time()
            response, parsed_json = zero_shot_assessment(
                question, student_answer, model_answer, model, tokenizer
            )
            record_time = time.time() - record_start

            # Extract LLM scores
            if parsed_json:
                m_c = int(parsed_json.get('Clarity_Score', 0))
                m_t = int(parsed_json.get('Terminology_Score', 0))
                m_v = int(parsed_json.get('Coverage_Score', 0))
                m_a = int(parsed_json.get('Accuracy_Score', 0))
                m_ot = bool(parsed_json.get('Off_Topic', False))
                llm_grade = (m_c + m_t + m_v + m_a) / 2
            else:
                m_c = m_t = m_v = m_a = 0
                m_ot = False
                llm_grade = 0

            # Track accuracy
            diff = abs(llm_grade - human_grade)
            if diff <= 1.0:
                correct_1 += 1
            if diff <= 0.5:
                correct_05 += 1

            acc_1 = correct_1 / len(results) * 100 if results else 0
            acc_05 = correct_05 / len(results) * 100 if results else 0

            pbar.set_description(f"🚀 {rate:.1f}/sec | ±1.0:{acc_1:.1f}% ±0.5:{acc_05:.1f}% | {student_id}")

            # Store result
            result = {
                'student_id': student_id,
                'Human_Clarity': h_c,
                'Human_Terminology': h_t,
                'Human_Coverage': h_v,
                'Human_Accuracy': h_a,
                'Human_Final': human_grade,
                'Human_Off_Topic': h_ot,
                'LLM_Clarity': m_c,
                'LLM_Terminology': m_t,
                'LLM_Coverage': m_v,
                'LLM_Accuracy': m_a,
                'LLM_Final': llm_grade,
                'LLM_Off_Topic': m_ot,
                'Diff': diff,
                'Within_1': diff <= 1.0,
                'Within_05': diff <= 0.5,
                'processing_time': record_time,
                'raw_response': response[:500] if response else ''
            }

            results.append(result)

            # Auto-save progress
            if len(results) % save_every == 0 and output_path:
                temp_df = pd.DataFrame(results)
                temp_df.to_csv(output_path, index=False)
                print(f"\n💾 Progress saved: {len(results)} records")

        except Exception as e:
            print(f"\n❌ Error at {idx} ({student_id}): {e}")
            continue

    # Final save
    final_df = pd.DataFrame(results)
    if output_path:
        final_df.to_csv(output_path, index=False)
        print(f"\n✅ Final results saved: {output_path}")

    # Summary
    print(f"\n{'='*60}")
    print("📊 BATCH PROCESSING COMPLETE")
    print(f"{'='*60}")
    print(f"Total processed: {len(results)}")
    print(f"±1.0 Accuracy: {correct_1/len(results)*100:.1f}%")
    print(f"±0.5 Accuracy: {correct_05/len(results)*100:.1f}%")

    # Off-topic summary
    ot_human = sum(1 for r in results if r['Human_Off_Topic'])
    ot_llm = sum(1 for r in results if r['LLM_Off_Topic'])
    print(f"Off-Topic (Human): {ot_human} ({ot_human/len(results)*100:.1f}%)")
    print(f"Off-Topic (LLM):   {ot_llm} ({ot_llm/len(results)*100:.1f}%)")

    return final_df

def process_in_chunks(df: pd.DataFrame, model, tokenizer,
                     chunk_size: int = 100, output_dir: str = None) -> pd.DataFrame:
    """Process large dataset in chunks"""

    total = len(df)
    num_chunks = (total + chunk_size - 1) // chunk_size

    all_results = []

    for i in range(num_chunks):
        start_idx = i * chunk_size
        end_idx = min((i + 1) * chunk_size, total)

        print(f"\n📦 CHUNK {i+1}/{num_chunks} (Records {start_idx}-{end_idx-1})")

        output_path = f"{output_dir}/chunk_{i+1}.csv" if output_dir else None

        chunk_results = batch_processing(
            df, model, tokenizer,
            start_idx=start_idx,
            end_idx=end_idx,
            save_every=50,
            output_path=output_path
        )

        all_results.append(chunk_results)

        torch.cuda.empty_cache()

    final_df = pd.concat(all_results, ignore_index=True)
    return final_df

print("✅ Batch processing functions loaded with Off-Topic detection!")

✅ Batch processing functions loaded with Off-Topic detection!


In [10]:
# Cell 10: Execute Batch Processing with ±1 Accuracy
print("🚀 EXECUTING BATCH PROCESSING")
print("="*60)

# Check if processing setup was completed in Cell 8
if 'PROCESSING_DF' not in locals() or PROCESSING_DF is None:
    print("❌ No processing setup found!")
    print("💡 Please run Cell 8 first to configure your processing scope")
    print("✅ Cell 10: Execution skipped - setup required")

else:
    print(f"✅ Processing setup confirmed:")
    print(f"   Model: {MODEL_CONFIGS[SELECTED_MODEL]['name']}")
    print(f"   Records: {len(PROCESSING_DF):,}")
    print(f"   Output: {OUTPUT_FILE}")

    # Final confirmation for large datasets
    if len(PROCESSING_DF) > 500:
        final_confirm = input(f"\n⚠️  About to process {len(PROCESSING_DF):,} records. This will take significant time. Proceed? (y/n): ").strip().lower()
        if final_confirm != 'y':
            print("❌ Processing cancelled by user")
            print("✅ Cell 10: Execution cancelled")
        else:
            proceed = True
    else:
        proceed = True

    if proceed:
        print(f"\n🎬 STARTING PROCESSING...")
        print(f"📅 Started at: {time.strftime('%Y-%m-%d %H:%M:%S')}")

        try:
            # Choose processing method based on dataset size
            if len(PROCESSING_DF) <= 500:
                # Direct processing for smaller datasets
                print(f"📋 Using direct processing for {len(PROCESSING_DF)} records")

                final_results = batch_processing(
                    df=PROCESSING_DF,
                    model=model,
                    tokenizer=tokenizer,
                    save_every=50,
                    output_path=OUTPUT_FILE
                )

            else:
                # Chunked processing for larger datasets
                print(f"📦 Using chunked processing for {len(PROCESSING_DF)} records")

                final_results = process_in_chunks(
                    df=PROCESSING_DF,
                    model=model,
                    tokenizer=tokenizer,
                    chunk_size=BATCH_SIZE
                )

                # Save final combined results
                final_results.to_csv(OUTPUT_FILE, index=False)

            # Processing completed successfully
            if len(final_results) > 0:
                print(f"\n🎊 PROCESSING COMPLETE!")
                print(f"📅 Completed at: {time.strftime('%Y-%m-%d %H:%M:%S')}")

                # Calculate basic statistics
                valid_data = final_results.dropna(subset=['LLM_Final', 'Human_Final'])

                print(f"\n📈 FINAL STATISTICS:")
                print(f"   Total processed: {len(final_results):,}")
                print(f"   Valid results: {len(valid_data):,}")
                print(f"   Success rate: {len(valid_data)/len(final_results)*100:.1f}%")

                if len(valid_data) > 10:
                    # Calculate ±1 accuracy
                    differences = abs(valid_data['LLM_Final'] - valid_data['Human_Final'])
                    within_1 = len(differences[differences <= 1.0])
                    within_05 = len(differences[differences <= 0.5])
                    exact_match = len(differences[differences == 0.0])

                    print(f"\n🎯 ACCURACY ANALYSIS:")
                    print(f"   Exact Match (±0.0):  {exact_match:4d} ({exact_match/len(valid_data)*100:5.1f}%)")
                    print(f"   Within ±0.5:         {within_05:4d} ({within_05/len(valid_data)*100:5.1f}%)")
                    print(f"   Within ±1.0:         {within_1:4d} ({within_1/len(valid_data)*100:5.1f}%)")

                    # Basic correlation
                    correlation = valid_data['LLM_Final'].corr(valid_data['Human_Final'])
                    mean_error = differences.mean()

                    print(f"\n📊 PERFORMANCE METRICS:")
                    print(f"   Correlation: {correlation:.3f}")
                    print(f"   Mean Abs Error: {mean_error:.3f}")
                    print(f"   LLM Avg Grade: {valid_data['LLM_Final'].mean():.2f}")
                    print(f"   Human Avg Grade: {valid_data['Human_Final'].mean():.2f}")

                # Off-topic analysis
                    print(f"\n🔍 OFF-TOPIC ANALYSIS:")

                print(f"\n💾 SAVED TO: {OUTPUT_FILE}")

            else:
                print(f"\n❌ No results generated - check for errors")

        except Exception as e:
            print(f"\n❌ Processing failed with error: {e}")
            print(f"💡 Check your model, data, and try again")

    print(f"\n✅ Cell 10: Batch processing execution complete!")

🚀 EXECUTING BATCH PROCESSING
✅ Processing setup confirmed:
   Model: DeepSeek-7B-Chat
   Records: 2,550
   Output: /content/drive/MyDrive/results_DeepSeek-7B-Chat_full_dataset.csv

⚠️  About to process 2,550 records. This will take significant time. Proceed? (y/n): y

🎬 STARTING PROCESSING...
📅 Started at: 2026-02-19 18:51:32
📦 Using chunked processing for 2550 records

📦 CHUNK 1/26 (Records 0-99)
🚀 Batch Processing: 100 records
💾 Auto-saving every 50 records
🧹 GPU cache cleared before processing


🚀 1.0/sec | ±1.0:48.5% ±0.5:34.3% | Q4 Student10: 100%|██████████| 100/100 [01:38<00:00,  1.01it/s]



📊 BATCH PROCESSING COMPLETE
Total processed: 100
±1.0 Accuracy: 48.0%
±0.5 Accuracy: 34.0%
Off-Topic (Human): 16 (16.0%)
Off-Topic (LLM):   34 (34.0%)

📦 CHUNK 2/26 (Records 100-199)
🚀 Batch Processing: 100 records
💾 Auto-saving every 50 records
🧹 GPU cache cleared before processing


🚀 0.9/sec | ±1.0:35.4% ±0.5:17.2% | Q7 Student20: 100%|██████████| 100/100 [01:53<00:00,  1.14s/it]



📊 BATCH PROCESSING COMPLETE
Total processed: 100
±1.0 Accuracy: 35.0%
±0.5 Accuracy: 17.0%
Off-Topic (Human): 5 (5.0%)
Off-Topic (LLM):   25 (25.0%)

📦 CHUNK 3/26 (Records 200-299)
🚀 Batch Processing: 100 records
💾 Auto-saving every 50 records
🧹 GPU cache cleared before processing


...


📊 BATCH PROCESSING COMPLETE
Total processed: 100
±1.0 Accuracy: 38.0%
±0.5 Accuracy: 28.0%
Off-Topic (Human): 19 (19.0%)
Off-Topic (LLM):   40 (40.0%)

📦 CHUNK 25/26 (Records 2400-2499)
🚀 Batch Processing: 100 records
💾 Auto-saving every 50 records
🧹 GPU cache cleared before processing


🚀 0.7/sec | ±1.0:47.5% ±0.5:27.3% | Q84 Student10: 100%|██████████| 100/100 [02:15<00:00,  1.35s/it]



📊 BATCH PROCESSING COMPLETE
Total processed: 100
±1.0 Accuracy: 47.0%
±0.5 Accuracy: 27.0%
Off-Topic (Human): 9 (9.0%)
Off-Topic (LLM):   29 (29.0%)

📦 CHUNK 26/26 (Records 2500-2549)
🚀 Batch Processing: 50 records
💾 Auto-saving every 50 records
🧹 GPU cache cleared before processing


🚀 1.0/sec | ±1.0:51.0% ±0.5:36.7% | Q85 Student30: 100%|██████████| 50/50 [00:51<00:00,  1.04s/it]


📊 BATCH PROCESSING COMPLETE
Total processed: 50
±1.0 Accuracy: 50.0%
±0.5 Accuracy: 36.0%
Off-Topic (Human): 6 (12.0%)
Off-Topic (LLM):   15 (30.0%)

🎊 PROCESSING COMPLETE!
📅 Completed at: 2026-02-19 19:37:23

📈 FINAL STATISTICS:
   Total processed: 2,550
   Valid results: 2,550
   Success rate: 100.0%

🎯 ACCURACY ANALYSIS:
   Exact Match (±0.0):   491 ( 19.3%)
   Within ±0.5:          706 ( 27.7%)
   Within ±1.0:         1086 ( 42.6%)

📊 PERFORMANCE METRICS:
   Correlation: 0.560
   Mean Abs Error: 1.589

❌ Processing failed with error: 'llm_final_grade'
💡 Check your model, data, and try again

✅ Cell 10: Batch processing execution complete!
